# Compare WCSim simulation vs. real WCTE data

Template notebook comparing the flattened simulation (`flatten_wcsim.C` output) against
merged real WCTE data (`WCTE_merged_production_R*.root`).

Because the simulation flattening was updated to mirror the real data schema
(`hit_pmt_charges`, `hit_pmt_calibrated_times`, `hit_mpmt_slot_ids`, `hit_pmt_position_ids`,
WCTE-frame positions, ...), most of the code below is written generically and works
on **either** file with the same branch names.

Sections:
1. Load the simulation
2. Load data and select muons the standard way (`analysis_tools.DataLoader` + `BeamSelection`,
   as in `analysis_examples/Example Using Data Loader.ipynb`) so the comparison is apples-to-apples
   with the simulated muon sample
3. Apply the same `good_wcte_pmts` channel mask to both
4. Basic comparison: number of hits per event
5. Per-event track summary table for the simulation (decayed / captured / scattered / ranged out / exited...)
6. Select simulated events by interaction type (e.g. muons that underwent a hard scatter)
7. Side-by-side event display (data vs. simulation, simulation hits colored by parent track)


## 0. Imports & configuration

In [ ]:
import sys
import numpy as np
import awkward as ak
import uproot
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from analysis_tools import DataLoader, BeamSelection, print_cherenkov_thresholds

# --- file paths: edit these for your own comparison ---
SIM_FILE  = "/eos/user/a/acraplet/WCSim/wcsim-wcte-easy/notebooks/flat_100MeV_30cm_0000.root"
DATA_FILE = "/eos/user/a/acraplet/WCSim/Analysis/data/WCTE_merged_production_R1419.root"

# WCTE_event_display (https://github.com/cookl/WCTE_event_display) - standalone repo,
# not part of the analysis_tools package, so it needs to be added to sys.path by hand.
EVENT_DISPLAY_DIR = "/eos/user/a/acraplet/WCTE_event_display"

# stop collecting data-side muons once we have this many events - the real data
# file has ~10^5 windows and a ~1 GB waveform branch we don't need here; raise
# this (or remove the early-stop below) once you've adapted the notebook further
N_DATA_MUON_EVENTS = 2000


## 1. Load the simulation

In [ ]:
sim_branches = [
    # per-hit
    "hit_pmt_charges", "hit_pmt_calibrated_times", "hit_mpmt_slot_ids", "hit_pmt_position_ids",
    "hit_track_id", "hit_x", "hit_y", "hit_z",
    # per-event truth
    "n_digihits", "event_number", "true_pdg", "true_E", "true_ke",
    "true_vtx_x", "true_vtx_y", "true_vtx_z", "true_start_x", "true_start_y", "true_start_z",
    "true_stop_x", "true_stop_y", "true_stop_z",
    "stop_process", "had_inelastic", "had_elastic", "n_prim_daughters",
    "true_exit_ke", "true_stopvol", "n_elastic", "n_inelastic",
    # per-track (jagged, one entry per saved track in the event)
    "track_id", "track_parent_id", "track_pdg", "track_process", "track_ke",
    "track_start_x", "track_start_y", "track_start_z",
    "track_end_x", "track_end_y", "track_end_z", "track_nhits",
]

with uproot.open(SIM_FILE) as f:
    sim = f["hits"].arrays(sim_branches, library="ak")

print(f"Loaded {len(sim)} simulated events from {SIM_FILE}")


## 2. Load data and select muons the standard way

This follows the exact pattern used in `analysis_examples/Example Using Data Loader.ipynb`:
`DataLoader` handles the window/hit quality-mask cuts, the per-run ACT/TOF cut values come
from the `vme_analysis_scalar_results` tree, and `BeamSelection` combines them into a PID cut.
We only request the branches we actually need - the file also has a ~1 GB waveform branch
we want `DataLoader` to skip entirely.


In [ ]:
loader = DataLoader(DATA_FILE, branches_to_load=[
    "hit_pmt_charges", "hit_pmt_calibrated_times", "hit_mpmt_slot_ids", "hit_pmt_position_ids",
    "event_number", "run_id",
    "vme_act_eveto", "vme_act_tagger", "vme_tof_corr",
])

vme_run_info = loader.get_vme_analysis_run_info()
print_cherenkov_thresholds(vme_run_info)


In [ ]:
vme_scalar_results = loader.get_vme_analysis_scalar_results()

tof_cut = vme_scalar_results["proton_tof_cut"]
if tof_cut == 0:
    print("WARNING: TOF separation unavailable for this run, setting TOF cut to default value of 999 ns.")
    tof_cut = 999

eveto_cut  = vme_scalar_results["act_eveto_cut"]
tagger_cut = vme_scalar_results["act_tagger_cut"] * 0

# MUONS: below threshold in the upstream ACT (act_eveto), above threshold in the
# downstream ACT (act_tagger) - the same definition used across the analysis_tools examples.
muon_sel = BeamSelection.selection(
    "muon",
    ["act_eveto",  "<", eveto_cut],
    ["act_tagger", ">", tagger_cut],
    ["tof",        "<", tof_cut],
)
muon_sel.describe()


In [ ]:
loader.apply_mPMT_data_quality_cuts()   # window_data_quality_mask==0 & hit_pmt_readout_mask==0
loader.apply_vme_event_quality_cuts()   # vme_digi_issues_bitmask==0 & vme_evt_quality_bitmask==0

muon_batches = []
n_muons = 0
for batch in loader.iterate(verbose=False, step_size="100 MB"):
    muon_batch = batch[muon_sel.mask(batch)]
    if len(muon_batch):
        muon_batches.append(muon_batch)
        n_muons += len(muon_batch)
    if n_muons >= N_DATA_MUON_EVENTS:
        break

data = ak.concatenate(muon_batches)
good_wcte_pmts = ak.to_numpy(loader.get_configuration()["good_wcte_pmts"])

print(f"Selected {len(data)} muon events from data (standard ACT/TOF-based PID cut)")


## 3. Apply the same `good_wcte_pmts` channel mask to both simulation and data

`good_wcte_pmts` (from the data file's `Configuration` tree) is a **run-level, static**
whitelist of `(mPMT slot, PMT position)` channels that were reading out reliably for the
whole run - encoded as `slot*100 + position`. It is *not* the same thing as the per-hit
`hit_pmt_readout_mask` already applied above. Applying it to the simulation too means both files are being compared over the same *live* set of channels, rather than sim seeing PMTs that were actually dead
in the real detector for this run.


In [ ]:
def apply_good_pmt_mask(events, good_encoded, extra_fields=()):
    """Keep only hits whose (slot, position) is in `good_encoded` (slot*100+position).

    `events` must be an awkward record array with hit_mpmt_slot_ids / hit_pmt_position_ids
    and the other per-hit fields you want filtered in lockstep (charges, times, ...).
    Works identically on the sim and data records since they share branch names.
    """
    hits = ak.zip({
        "slot": events["hit_mpmt_slot_ids"],
        "pos":  events["hit_pmt_position_ids"],
        "q":    events["hit_pmt_charges"],
        "t":    events["hit_pmt_calibrated_times"],
        **{name: events[name] for name in extra_fields},
    })

    encoded = hits.slot * 100 + hits.pos
    # np.isin needs flat arrays; unflatten back to the original jagged structure after
    flat_mask = np.isin(ak.flatten(encoded).to_numpy(), good_encoded)
    good_mask = ak.unflatten(flat_mask, ak.num(encoded))

    return hits[good_mask]

good_encoded = good_wcte_pmts  # already slot*100+position, straight from Configuration

sim_hits_masked  = apply_good_pmt_mask(sim,  good_encoded, extra_fields=("hit_track_id",))
data_hits_masked = apply_good_pmt_mask(data, good_encoded)

print(f"good_wcte_pmts covers {len(good_encoded)} channels")
print(f"sim:  mean hits/event before mask = {ak.mean(ak.num(sim['hit_pmt_charges'])):.1f}, "
      f"after = {ak.mean(ak.num(sim_hits_masked['q'])):.1f}")
print(f"data: mean hits/event before mask = {ak.mean(ak.num(data['hit_pmt_charges'])):.1f}, "
      f"after = {ak.mean(ak.num(data_hits_masked['q'])):.1f}")


## 4. Basic comparison: number of hits per event

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

bins = np.linspace(0, 700, 41)
for ax, sim_n, data_n, title in [
    (axes[0], ak.num(sim["hit_pmt_charges"]), ak.num(data["hit_pmt_charges"]), "raw (no good-PMT mask)"),
    (axes[1], ak.num(sim_hits_masked["q"]),   ak.num(data_hits_masked["q"]),   "good_wcte_pmts-masked"),
]:
    ax.hist(ak.to_numpy(sim_n),  bins=bins, density=True, histtype="step", lw=2, label="simulation")
    ax.hist(ak.to_numpy(data_n), bins=bins, density=True, histtype="step", lw=2, label="data")
    ax.set_xlabel("digitized hits / event")
    ax.set_ylabel("normalized")
    ax.set_title(title)
    ax.legend()

fig.tight_layout()


## 5. Per-event track summary for the simulation

Classifies each simulated event's primary by its ultimate fate, using the
`stop_process` / `had_elastic` / `had_inelastic` / `true_exit_ke` truth branches
written by `flatten_wcsim.C`.


In [ ]:
def classify_outcome(stop_process, had_inelastic, had_elastic, true_exit_ke):
    """Human-readable summary of what happened to the primary, from MC truth."""
    if "Decay" in stop_process:
        return "decayed"
    if "CaptureAtRest" in stop_process:
        return "captured at rest"
    if had_inelastic:
        return "hadronic inelastic interaction"
    if had_elastic:
        return "hadronic elastic scatter"
    if true_exit_ke >= 0:
        return "exited the tank"
    return "ranged out (ionization only)"

outcomes = [
    classify_outcome(sp, hi, he, ke)
    for sp, hi, he, ke in zip(
        ak.to_list(sim["stop_process"]), ak.to_list(sim["had_inelastic"]),
        ak.to_list(sim["had_elastic"]), ak.to_list(sim["true_exit_ke"]),
    )
]

track_summary = pd.DataFrame({
    "event_number":     ak.to_numpy(sim["event_number"]),
    "true_pdg":         ak.to_numpy(sim["true_pdg"]),
    "true_E_MeV":       ak.to_numpy(sim["true_E"]),
    "n_tracks":         ak.to_numpy(ak.num(sim["track_id"])),
    "n_prim_daughters": ak.to_numpy(sim["n_prim_daughters"]),
    "n_elastic":        ak.to_numpy(sim["n_elastic"]),
    "n_inelastic":      ak.to_numpy(sim["n_inelastic"]),
    "stop_process":     ak.to_list(sim["stop_process"]),
    "outcome":          outcomes,
})

track_summary.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
track_summary["outcome"].value_counts().plot.barh(ax=ax)
ax.set_xlabel("events")
ax.set_title("Simulated primary-particle outcome")
fig.tight_layout()


## 6. Select simulated events by interaction type

Example: find muons (`|pdg| == 13`) that underwent a **hard scatter** - i.e. produced
at least one direct daughter via something other than continuous ionization
(`muIoni`) or plain transport. This naturally picks up hadronic elastic/inelastic
scatters, muon-nuclear interactions, bremsstrahlung, pair production, decay, and
capture-at-rest, without having to hardcode every possible Geant4 process name.

The primary's own track is identified inside `track_id` by matching its start
position to `true_start_*` (bit-identical, since both come from the same
`WCSimRootTrack::GetStart()` call in `flatten_wcsim.C`).


In [ ]:
SOFT_PROCESSES = {"muIoni", "hIoni", "eIoni", "Transportation"}

def find_primary_index(ev):
    """Index into ev.track_* of the primary track (matched by start position)."""
    matches = np.flatnonzero(
        (np.asarray(ev["track_start_x"]) == ev["true_start_x"]) &
        (np.asarray(ev["track_start_y"]) == ev["true_start_y"]) &
        (np.asarray(ev["track_start_z"]) == ev["true_start_z"])
    )
    return int(matches[0]) if len(matches) else None

def is_hard_scatter(ev, pdg=13, soft_processes=SOFT_PROCESSES):
    if abs(ev["true_pdg"]) != pdg:
        return False
    prim_idx = find_primary_index(ev)
    if prim_idx is None:
        return False
    primary_track_id = ev["track_id"][prim_idx]
    daughter_processes = [
        proc for tid, proc in zip(ev["track_parent_id"], ev["track_process"])
        if tid == primary_track_id
    ]
    return any(proc not in soft_processes for proc in daughter_processes)

hard_scatter_mask = np.array([is_hard_scatter(ev) for ev in sim])
hard_scatter_events = sim[hard_scatter_mask]

print(f"{hard_scatter_mask.sum()} / {len(sim)} muon events classified as 'hard scatter'")
track_summary[hard_scatter_mask][["event_number", "stop_process", "outcome", "n_prim_daughters"]]


## 7. Side-by-side event display: data vs. simulation

Uses `WCTE_event_display` (https://github.com/cookl/WCTE_event_display), a standalone
2D "unrolled cylinder" display keyed on `(mPMT slot, PMT position)` channels - exactly
the convention `hit_mpmt_slot_ids` / `hit_pmt_position_ids` already use on both the
data and (post-`flatten_wcsim.C`-update) simulation side, so no tube-number mapping
step is needed for either file.

`EventDisplay.plotEventDisplay(...)` always creates its own figure, so for a genuine
side-by-side layout we reuse its lower-level pieces (`coordinates_eachChannel`,
`process_data`, `mPMT_2D_projection`) and draw into our own `plt.subplots(1, 2)` axes.


In [ ]:
sys.path.insert(0, EVENT_DISPLAY_DIR)
from EventDisplay import EventDisplay  # noqa: E402
import os
os.chdir(EVENT_DISPLAY_DIR)  # load_mPMT_positions resolves its CSV relative to the module dir

ed = EventDisplay()
ed.load_mPMT_positions("mPMT_2D_projection_angles.csv")


In [ ]:
def channel_first_value(slots, positions, values, n_channels):
    """One representative value per (slot, position) channel - the first hit seen.

    Unlike EventDisplay.process_data (sum, or min with a 0-vs-unset ambiguity), this
    is what we want for a *categorical* value like a track id: pick a single
    representative hit per channel rather than summing/averaging track ids.
    """
    out = np.full(n_channels, np.nan)
    for slot, pos, v in zip(slots, positions, values):
        ch = 19 * int(slot) + int(pos)
        if np.isnan(out[ch]):
            out[ch] = v
    return out

def classify_mpmt_slots(good_encoded, n_mpmts=106, n_pos=19):
    """Classify each mPMT slot by how many of its 19 positions are in the
    good_wcte_pmts whitelist (slot*100+position encoding).

    Returns (fully_off_slots, partially_off_slots): slots with 0/19 good
    channels, and slots with some-but-not-all good channels, respectively.
    Fully-good slots (19/19) are not returned - nothing to highlight there.
    """
    good_set = set(int(v) for v in good_encoded)
    fully_off, partially_off = [], []
    for slot in range(n_mpmts):
        n_good = sum((slot * 100 + pos) in good_set for pos in range(n_pos))
        if n_good == 0:
            fully_off.append(slot)
        elif n_good < n_pos:
            partially_off.append(slot)
    return np.array(fully_off), np.array(partially_off)

def draw_event(ax, ed, data, cmap, norm, title, color_label=None,
                fully_off_slots=(), partially_off_slots=()):
    """Minimal reimplementation of EventDisplay.plotEventDisplay that draws into a
    given ax instead of creating its own figure, so two events can share one row.

    fully_off_slots / partially_off_slots (mPMT slot IDs, see classify_mpmt_slots)
    are drawn as extra highlighted circles on top of the normal per-mPMT outline,
    so it's visually obvious which blank regions are due to the good_wcte_pmts
    mask rather than just this one event happening to have no hits there.
    """
    import copy
    from matplotlib.patches import Circle
    from matplotlib.collections import PatchCollection

    coordinates = ed.coordinates_eachChannel()
    plot_data = data.copy()
    plot_data[plot_data == 0] = np.nan  # keep unhit channels blank, as plotEventDisplay does

    cmap = copy.copy(cmap)
    cmap.set_bad(color="white")

    pmt_circles = [Circle((x, y), radius=0.48) for x, y in ed.mPMT_2D_projection[:, 1:3]]
    ax.add_collection(PatchCollection(pmt_circles, facecolor="none", linewidths=1, edgecolors="0.85"))
    pts = ax.scatter(coordinates[:, 0], coordinates[:, 1], c=plot_data, s=25, cmap=cmap, norm=norm)

    # highlight mPMTs excluded (fully or partially) by the good_wcte_pmts mask
    if len(fully_off_slots):
        xy = ed.mPMT_2D_projection[np.asarray(fully_off_slots), 1:3]
        ax.add_collection(PatchCollection(
            [Circle((x, y), radius=0.75) for x, y in xy],
            facecolor="none", edgecolors="red", linewidths=2, zorder=5))
    if len(partially_off_slots):
        xy = ed.mPMT_2D_projection[np.asarray(partially_off_slots), 1:3]
        ax.add_collection(PatchCollection(
            [Circle((x, y), radius=0.75) for x, y in xy],
            facecolor="none", edgecolors="orange", linewidths=2, linestyles="--", zorder=5))

    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.colorbar(pts, ax=ax, pad=0.01, label=color_label)
    return pts


In [ ]:
# --- classify which mPMT slots the good_wcte_pmts mask excludes (once, reused below) ---
fully_off_slots, partially_off_slots = classify_mpmt_slots(good_wcte_pmts)
print(f"{len(fully_off_slots)} mPMT slots fully off, {len(partially_off_slots)} partially off "
      f"(good_wcte_pmts covers {len(good_wcte_pmts)}/{106*19} channels)")


In [ ]:
# --- pick one data event and one simulated ("hard scatter") event to compare ---
# NOTE: plot the good_wcte_pmts-MASKED hits (data_hits_masked / sim_hits_masked from
# section 3), not the raw data/sim events - otherwise the two panels aren't actually
# looking at the same set of live channels, which is what made the "off" mPMTs look
# different between data and simulation in the first place.
data_event_idx = 20
sim_event_idx  = int(np.flatnonzero(hard_scatter_mask)[0]) if hard_scatter_mask.any() else 3

data_ev, data_ev_hits = data[data_event_idx], data_hits_masked[data_event_idx]
sim_ev,  sim_ev_hits  = sim[sim_event_idx],   sim_hits_masked[sim_event_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- left: real data event, colored by charge ---
data_channel_q = channel_first_value(
    ak.to_list(data_ev_hits["slot"]), ak.to_list(data_ev_hits["pos"]),
    ak.to_list(data_ev_hits["q"]), ed.nChannels,
)
draw_event(
    axes[0], ed, np.nan_to_num(data_channel_q), cmap=plt.cm.plasma,
    norm=mcolors.Normalize(vmin=0, vmax=np.nanmax(data_channel_q) if np.any(~np.isnan(data_channel_q)) else 1),
    title=f"data: run {int(data_ev['run_id'])}, event {int(data_ev['event_number'])}",
    color_label="charge [p.e.]",
    fully_off_slots=fully_off_slots, partially_off_slots=partially_off_slots,
)

# --- right: simulated event, colored by parent TRACK ID (not charge) ---
track_ids_in_event = sorted(set(ak.to_list(sim_ev_hits["hit_track_id"])))
track_id_to_color_index = {tid: i for i, tid in enumerate(track_ids_in_event)}
hit_color_index = [track_id_to_color_index[tid] for tid in ak.to_list(sim_ev_hits["hit_track_id"])]

sim_channel_track = channel_first_value(
    ak.to_list(sim_ev_hits["slot"]), ak.to_list(sim_ev_hits["pos"]),
    hit_color_index, ed.nChannels,
)
draw_event(
    axes[1], ed, sim_channel_track, cmap=plt.cm.tab20,
    norm=mcolors.Normalize(vmin=0, vmax=max(len(track_ids_in_event) - 1, 1)),
    title=f"simulation: event {int(sim_ev['event_number'])} ({classify_outcome(str(sim_ev['stop_process']), sim_ev['had_inelastic'], sim_ev['had_elastic'], sim_ev['true_exit_ke'])})",
    color_label="track index (see legend below)",
    fully_off_slots=fully_off_slots, partially_off_slots=partially_off_slots,
)

# proxy legend entries for the red/orange highlight circles (PatchCollections don't auto-legend)
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", color="red", markerfacecolor="none", linestyle="", markersize=12, label="mPMT fully off (good_wcte_pmts)"),
    Line2D([0], [0], marker="o", color="orange", markerfacecolor="none", linestyle="", markersize=12, label="mPMT partially off (good_wcte_pmts)"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2)
fig.tight_layout()

# legend: which track index corresponds to which particle
print("Track color legend for the simulated event:")
for tid, idx in track_id_to_color_index.items():
    matches = np.flatnonzero(np.asarray(sim_ev["track_id"]) == tid)
    if len(matches):
        j = matches[0]
        tag = " <- PRIMARY" if tid == sim_ev["track_id"][find_primary_index(sim_ev)] else ""
        print(f"  color {idx}: track_id={tid}  pdg={sim_ev['track_pdg'][j]}  "
              f"process={sim_ev['track_process'][j]}  ke={sim_ev['track_ke'][j]:.2f} MeV{tag}")
    else:
        print(f"  color {idx}: track_id={tid}  (dark noise / not in truth track list)" if tid == -1
              else f"  color {idx}: track_id={tid}  (no truth match)")
